In [1]:

%load_ext ElasticNotebook
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_19_try_0.pickle

trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['add_gap_rows']
me:  20
trying: ['mylen']
me:  12
trying: ['txt_file']
me:  12
trying: ['sample_submission']
me:  1
trying: ['total_gaps']
me:  19
trying: ['train_last']
me:  10
trying: ['train_first']
me:  10
trying: ['train_first_last']
me:  10
trying: ['sample_discourse_id']
me:  2
trying: ['np']
me:  0
trying: ['train']
me:  13
trying: ['train_txt']
me:  1
trying: ['cols_to_display']
me:  14
trying: ['IREWR_tmp2']
me:  17
trying: ['tqdm']
me:  0
trying: ['IREWR_plug_1']
me:  17
trying: ['counter']
me:  11
trying: ['glob']
me:  0
trying: ['test_txt']
me:  1
trying: ['ax1']
me:  7
trying: ['ax2']
me:  7
trying: ['IREWR_tmp']
me:  16
trying: ['last_ones']
me:  13
trying: ['IREWR_plug_2']
me:  16
trying: ['stopwords']
me:  1
trying: ['cols_to_merge']
me:  13
trying: ['pd']
me:  0
trying: ['style']
me:  0
trying: ['FuncFormatter']
me:  0
trying: ['CountVectorizer']
me:  0
trying: ['ax']
me:  8
trying:

me:  12
Declaring variable np
Declaring variable tqdm
Declaring variable glob
Declaring variable pd
Declaring variable style
Declaring variable FuncFormatter
Declaring variable CountVectorizer
Declaring variable plt
Declaring variable warnings
Declaring variable sample_submission
Declaring variable train_txt
Declaring variable test_txt
Declaring variable stopwords
Declaring variable sample_discourse_id
Declaring variable ax1
Declaring variable ax2
Declaring variable ax
Declaring variable av_per_essay
Declaring variable train_last
Declaring variable train_first
Declaring variable train_first_last
Declaring variable i_1
Declaring variable i_1
Declaring variable counter
Declaring variable mylen
Declaring variable txt_file
Declaring variable word_dict
Declaring variable myword
Declaring variable data
Declaring variable myid
Declaring variable t
Declaring variable len_dict
Declaring variable i_3
Declaring variable i_3
Declaring variable train
Declaring variable last_ones
Declaring variable 

In [3]:
%%RecordEvent
%%time
### cell 20 ###

def add_gap_rows_2(essay):
    # select only the relevant columns for this essay
    cols_to_keep = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    df = train.loc[train['id'] == essay, cols_to_keep].reset_index(drop=True)
    # preserve original print
    print(df)

    # vectorized computation of interior gaps
    starts = df['discourse_end'].shift().add(1)
    ends = df['discourse_start'].sub(1)
    mask = (df['gap_length'] > 0) & (df.index >= 1)
    if mask.any():
        new_gaps = pd.DataFrame({
            'discourse_start': starts[mask],
            'discourse_end': ends[mask],
            'discourse_type': 'Nothing',
            'gap_length': np.nan,
            'gap_end_length': np.nan
        })
        df = pd.concat([df, new_gaps], ignore_index=True)

    # sort after inserting interior gaps
    df = df.sort_values('discourse_start').reset_index(drop=True)

    # add end gap if present (appended at bottom to match original semantics)
    if df['gap_end_length'].iloc[-1] > 0:
        start = df['discourse_end'].iloc[-1] + 1
        end = start + df['gap_end_length'].iloc[-1]
        df.loc[len(df)] = [start, end, 'Nothing', np.nan, np.nan]

    return df


def print_colored_essay(essay):
    df_essay = add_gap_rows_2(essay)
    # build ents list via vectorized to_dict
    ents = (
        df_essay[['discourse_start', 'discourse_end', 'discourse_type']]
        .rename(columns={
            'discourse_start': 'start',
            'discourse_end': 'end',
            'discourse_type': 'label'
        })
        .astype({'start': 'int', 'end': 'int'})
        .to_dict('records')
    )
    # read the essay text
    essay_file = (
        f"/home/dias-benchmarks/notebooks/erikbruin/"
        f"nlp-on-student-writing-eda/input/feedback-prize-2021/train/{essay}.txt"
    )
    with open(essay_file, 'r') as file:
        data = file.read()

    doc2 = {"text": data, "ents": ents}
    colors = {
        'Lead': '#EE11D0', 'Position': '#AB4DE1', 'Claim': '#1EDE71',
        'Evidence': '#33FAFA', 'Counterclaim': '#4253C1',
        'Concluding Statement': 'yellow', 'Rebuttal': 'red'
    }
    options = {"ents": df_essay.discourse_type.unique().tolist(), "colors": colors}


CPU times: user 3 µs, sys: 1 µs, total: 4 µs
Wall time: 7.39 µs


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_20_try_0.pickle

migration speed (bps): 748771909.6753403
---------------------------
variables to migrate:
ax 1210
myword 28
data 2813
myid 61
style 72
t 166
FuncFormatter 1072
all_gaps 288
sample_discourse_id 32
CountVectorizer 1072
len_dict 589920
pd 72
plt 72
warnings 72
glob 144
test_txt 120
total_gaps 1329
print_colored_essay 144
train_txt 126104
train_first 302
train_first_last 334
train_last 395
np 72
train 82800
tqdm 1072
i_3 28
last_ones 12567
cols_to_merge 120
cols_to_display 120
counter 28
add_gap_rows_2 144
i_1 28
IREWR_tmp2 930
IREWR_plug_1 61
ax1 536
sample_submission 569
ax2 536
stopwords 48
add_gap_rows 144
IREWR_tmp 916
IREWR_plug_2 61
mylen 28
txt_file 208
word_dict 589920
av_per_essay 1338
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_20_try_0.pickle


In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['train_txt', 'train']
Intermediate variables ['stopwords', 'test_txt', 'sample_submission']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['train']
Active variables ['sample_discourse_id']
Intermediate variables []
Future variables ['train_txt', 'train']
Modified dataframes
======= Cell 2 =======
Input variables ['train']
Active variables ['cols_to_display', 'train']
Intermediate variables []
Future variables ['sample_discourse_id', 'train_txt']
Modified dataframes
  train
    Input columns: set()
    Changed columns: set()
    Created columns: {'discourse_len', 'pred_len'}
    Deleted columns: set()
======= Cell 3 =======
Input variables ['cols_to_display', 'train']
Active variables []
Intermediate variables []
Future variables ['sample_discourse_id', 'train_txt', 'cols_to_display', 'train']
Modified dataframes
======= Cell 4 =======
Input variables ['sample_discourse_id', 'train']
Active vari

In [6]:
with open(
    "/home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_20_try_0.pkl",
    "wb",
) as f:
    pickle.dump(opt_cell_exec_info[20], f)

In [7]:
opt_output = Out.get(4)

In [8]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated_cpu/checkpoints/post_cell_20.pickle

import numpy as np
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output

trying: ['IREWR_plug_1']
me:  33
trying: ['IREWR_tmp2']
me:  33
trying: ['np']
me:  0
trying: ['tqdm']
me:  0
trying: ['i_1']
me:  21
trying: ['sample_submission']
me:  1
trying: ['counter']
me:  21
trying: ['IREWR_plug_2']
me:  31
trying: ['test_txt']
me:  1
trying: ['IREWR_tmp']
me:  31
trying: ['ax1']
me:  13
trying: ['ax2']
me:  13
trying: ['warnings']
me:  0
trying: ['style']
me:  0
trying: ['all_gaps']
me:  35
trying: ['add_gap_rows_2']
me:  41
trying: ['CountVectorizer']
me:  0
trying: ['plt']
me:  0
trying: ['FuncFormatter']
me:  0
trying: ['i_3']
me:  25
trying: ['stopwords']
me:  0
trying: ['print_colored_essay']
me:  41
trying: ['ax']
me:  15
trying: ['pd']
me:  0
trying: ['last_ones']
me:  25
trying: ['av_per_essay']
me:  15
trying: ['cols_to_merge']
me:  25
trying: ['glob']
me:  0
trying: ['total_gaps']
me:  37
trying: ['train_txt']
me:  1
trying: ['sample_discourse_id']
me:  3
trying: ['t']
me:  23
trying: ['myid']
me:  23
trying: ['myword']
me:  23
trying: ['orig_output'